In [1]:
from src.stats_dp.request_class import Count, Sum, Mean, Ratio, Quantile, InfoDataset
from src.stats_dp.pipeline_class import Pipeline
import seaborn as sns
import polars as pl

lf = pl.DataFrame(sns.load_dataset("penguins").dropna()).lazy()

info_dataset = InfoDataset(lf, contribution_individu_max=1, borne_max_taille_dataset=1)

print(lf)

naive plan: (run LazyFrame.explain(optimized=True) to see the optimized plan)

DF ["species", "island", "bill_length_mm", "bill_depth_mm"]; PROJECT */7 COLUMNS; SELECTION: None


## 1. Comptage

In [2]:
info_dataset.create_context_rho(budget_rho=0.1, list_poids=[1])

requete = Count(by=["species", "island"])

requete.execute(lf)
requete.execute_dp(info_dataset)

<class 'polars.lazyframe.frame.LazyFrame'>


count,species,island
i32,str,str
68,"""Chinstrap""","""Dream"""
-2,"""Gentoo""","""Dream"""
115,"""Gentoo""","""Biscoe"""
44,"""Adelie""","""Torgersen"""
43,"""Adelie""","""Biscoe"""
0,"""Chinstrap""","""Biscoe"""
0,"""Chinstrap""","""Torgersen"""
4,"""Gentoo""","""Torgersen"""
55,"""Adelie""","""Dream"""


## 2. Total

## 3. Moyenne

## 4. Quantile

# Utilisation du Pipeline

In [4]:
import seaborn as sns
import polars as pl

requetes = {
    "req_1": Count(),
    "req_2": Count(by=["species"]),
    "req_3": Count(by=["island", "species"]),
    "req_4": Count(by=["island", "sex", "species"]),
    "req_5": Count(by=["island", "sex"]),
    "req_6": Count(by=["sex"]),
    "req_7": Count(by=["island"]),
    "req_8": Count(by=["sex", "species"]),

    "req_9": Sum(variable="body_mass_g", bounds=[2700.0, 6300.0]),
    "req_10": Sum(variable="body_mass_g", bounds=[2700.0, 6300.0], by=["species"]),
    "req_11": Sum(variable="body_mass_g", bounds=[2700.0, 6300.0], by=["island", "species"]),
    "req_12": Sum(variable="body_mass_g", bounds=[2700.0, 6300.0], by=["island", "sex", "species"]),
    "req_13": Sum(variable="body_mass_g", bounds=[2700.0, 6300.0], by=["island", "sex"]),
    "req_14": Sum(variable="body_mass_g", bounds=[2700.0, 6300.0], by=["sex"]),
    "req_15": Sum(variable="body_mass_g", bounds=[2700.0, 6300.0], by=["sex", "species"]),
    "req_16": Sum(variable="body_mass_g", bounds=[2700.0, 6300.0], by=["island"]),

    "req_17": Mean(variable="body_mass_g", bounds=[2700.0, 6300.0]),
    "req_18": Mean(variable="body_mass_g", bounds=[2700.0, 6300.0], by=["species"]),
    "req_19": Mean(variable="body_mass_g", bounds=[2700.0, 6300.0], by=["island", "species"]),
    "req_20": Mean(variable="body_mass_g", bounds=[2700.0, 6300.0], by=["island", "sex", "species"]),
    "req_21": Mean(variable="body_mass_g", bounds=[2700.0, 6300.0], by=["island", "sex"]),
    "req_22": Mean(variable="body_mass_g", bounds=[2700.0, 6300.0], by=["sex"]),
    "req_23": Mean(variable="body_mass_g", bounds=[2700.0, 6300.0], by=["island"]),

    "req_24": Ratio(
        variable="body_mass_g", bounds=[2700.0, 6300.0],
        variable_denominateur="bill_length_mm", bounds_denominateur=[32.1, 59.6],
        by=["species"]
    ),
    "req_25": Ratio(
        variable="body_mass_g", bounds=[2700.0, 6300.0],
        variable_denominateur="bill_length_mm", bounds_denominateur=[32.1, 59.6],
        by=["island", "species"]
    ),
    "req_26": Ratio(
        variable="body_mass_g", bounds=[2700.0, 6300.0],
        variable_denominateur="bill_length_mm", bounds_denominateur=[32.1, 59.6],
        by=["island", "sex", "species"]
    ),
    "req_27": Ratio(
        variable="body_mass_g", bounds=[2700.0, 6300.0],
        variable_denominateur="bill_length_mm", bounds_denominateur=[32.1, 59.6],
        by=["island", "sex"]
    ),
    "req_28": Ratio(
        variable="body_mass_g", bounds=[2700.0, 6300.0],
        variable_denominateur="bill_length_mm", bounds_denominateur=[32.1, 59.6],
        by=["sex"]
    ),
    "req_29": Ratio(
        variable="body_mass_g", bounds=[2700.0, 6300.0],
        variable_denominateur="bill_length_mm", bounds_denominateur=[32.1, 59.6]
    ),

    "req_30": Quantile(
        variable="body_mass_g", bounds=[2700.0, 6300.0],
        alpha=[0.5], nb_candidats="300"
    ),
    "req_31": Quantile(
        variable="body_mass_g", bounds=[2700.0, 6300.0],
        by=["species"], alpha=[0.5], nb_candidats="300"
    ),
    "req_32": Quantile(
        variable="body_mass_g", bounds=[2700.0, 6300.0],
        by=["island", "sex", "species"], alpha=[0.5], nb_candidats="300"
    ),
    "req_33": Quantile(
        variable="body_mass_g", bounds=[2700.0, 6300.0],
        by=["island", "species"], alpha=[0.5], nb_candidats="300"
    )
}


dict_poids = {key: 1/len(requetes) for key in requetes.keys()}

lf = pl.DataFrame(sns.load_dataset("penguins").dropna()).lazy()

In [6]:
pipeline_request = Pipeline(requetes, info_dataset)
pipeline_request.execute()

==> Début execute
<== Fin execute (temps total : 0.02 secondes)


{'req_1': shape: (1, 1)
 ┌───────┐
 │ count │
 │ ---   │
 │ u32   │
 ╞═══════╡
 │ 333   │
 └───────┘,
 'req_2': shape: (3, 2)
 ┌───────────┬───────┐
 │ species   ┆ count │
 │ ---       ┆ ---   │
 │ str       ┆ u32   │
 ╞═══════════╪═══════╡
 │ Gentoo    ┆ 119   │
 │ Adelie    ┆ 146   │
 │ Chinstrap ┆ 68    │
 └───────────┴───────┘,
 'req_3': shape: (5, 3)
 ┌───────────┬───────────┬───────┐
 │ island    ┆ species   ┆ count │
 │ ---       ┆ ---       ┆ ---   │
 │ str       ┆ str       ┆ u32   │
 ╞═══════════╪═══════════╪═══════╡
 │ Biscoe    ┆ Gentoo    ┆ 119   │
 │ Biscoe    ┆ Adelie    ┆ 44    │
 │ Dream     ┆ Chinstrap ┆ 68    │
 │ Torgersen ┆ Adelie    ┆ 47    │
 │ Dream     ┆ Adelie    ┆ 55    │
 └───────────┴───────────┴───────┘,
 'req_4': shape: (10, 4)
 ┌───────────┬────────┬───────────┬───────┐
 │ island    ┆ sex    ┆ species   ┆ count │
 │ ---       ┆ ---    ┆ ---       ┆ ---   │
 │ str       ┆ str    ┆ str       ┆ u32   │
 ╞═══════════╪════════╪═══════════╪═══════╡
 │ Biscoe  

In [10]:
pipeline_request = Pipeline(requetes, info_dataset)
# pipeline_request.execute()
X = pipeline_request.precision_dp(budget_global=0.1, dict_poids=dict_poids)
print(X)

==> Début precision_dp
<== Fin precision_dp (temps total : 0.22 secondes)
{'Comptage': shape: (8, 4)
┌─────────┬──────────────────────────────┬───────────────────────┬──────────────────┐
│ requête ┆ groupement                   ┆ écart type estimation ┆ écart type bruit │
│ ---     ┆ ---                          ┆ ---                   ┆ ---              │
│ str     ┆ str                          ┆ f64                   ┆ f64              │
╞═════════╪══════════════════════════════╪═══════════════════════╪══════════════════╡
│ req_1   ┆ Aucun                        ┆ 4.1                   ┆ 6.4              │
│ req_2   ┆ species                      ┆ 4.2                   ┆ 6.4              │
│ req_3   ┆ ('island', 'species')        ┆ 4.0                   ┆ 6.4              │
│ req_4   ┆ ('island', 'sex', 'species') ┆ 4.0                   ┆ 6.4              │
│ req_5   ┆ ('island', 'sex')            ┆ 4.0                   ┆ 6.4              │
│ req_6   ┆ sex                        

In [12]:
pipeline_request = Pipeline(requetes, info_dataset)
X = pipeline_request.execute_dp(budget_global=0.1, dict_poids=dict_poids)
print(X)

==> Début execute_dp
✅ Context terminée en 0.05 secondes.
<== Fin execute_dp (temps total : 28.13 secondes)
{'req_1': shape: (1, 1)
┌───────┐
│ count │
│ ---   │
│ f64   │
╞═══════╡
│ 338.1 │
└───────┘, 'req_2': shape: (3, 2)
┌───────────┬───────┐
│ species   ┆ count │
│ ---       ┆ ---   │
│ str       ┆ f64   │
╞═══════════╪═══════╡
│ Adelie    ┆ 142.4 │
│ Chinstrap ┆ 72.7  │
│ Gentoo    ┆ 123.0 │
└───────────┴───────┘, 'req_3': shape: (9, 3)
┌───────────┬───────────┬───────┐
│ island    ┆ species   ┆ count │
│ ---       ┆ ---       ┆ ---   │
│ str       ┆ str       ┆ f64   │
╞═══════════╪═══════════╪═══════╡
│ Biscoe    ┆ Adelie    ┆ 43.8  │
│ Biscoe    ┆ Chinstrap ┆ 3.4   │
│ Biscoe    ┆ Gentoo    ┆ 119.2 │
│ Dream     ┆ Adelie    ┆ 55.5  │
│ Dream     ┆ Chinstrap ┆ 63.8  │
│ Dream     ┆ Gentoo    ┆ 3.8   │
│ Torgersen ┆ Adelie    ┆ 43.2  │
│ Torgersen ┆ Chinstrap ┆ 5.4   │
│ Torgersen ┆ Gentoo    ┆ 0.0   │
└───────────┴───────────┴───────┘, 'req_4': shape: (18, 4)
┌───────────┬────

In [ ]:
from src.request_class import Count

a = Count(by=["region", "age"])
b = Count(by=["age", "region"])
print(a.to_query_dict())
print(b.to_query_dict())
print(a == b)

{'type': 'Comptage', 'by': ['region', 'age']}
{'type': 'Comptage', 'by': ['age', 'region']}
True


In [15]:
import opendp.prelude as dp

dp.enable_features("contrib")


context_param = {
    "data": lf,
    "privacy_unit": dp.unit_of(contributions=1),
    "margins": [dp.polars.Margin(max_partition_length=1)],
}

context = dp.Context.compositor(
    **context_param,
    privacy_loss=dp.loss_of(rho=0.5),
    split_by_weights=[0.5, 0.5]
)

print(context)

Context(
    accountant = Measurement(
        input_domain   = FrameDomain(species: str, island: str, bill_length_mm: f64, bill_depth_mm: f64, flipper_length_mm: f64, body_mass_g: f64, sex: str; margins=[{}]),
        input_metric   = SymmetricDistance(),
        output_measure = ZeroConcentratedDivergence),
    d_in       = 1,
    d_mids     = [0.25, 0.25])


In [ ]:
import opendp.prelude as dp

dp.enable_features("contrib")

context_param = {
    "data": lf,
    "privacy_unit": dp.unit_of(contributions=1),
    "margins": [dp.polars.Margin(max_partition_length=1)],
}

context = dp.Context.compositor(
    **context_param,
    privacy_loss=dp.loss_of(rho=0.5),
    split_by_weights=[1, 3]
)
print(context)

context_2 = dp.Context.compositor(
    **context_param,
    privacy_loss=dp.loss_of(epsilon=0.5),
    split_by_weights=[1]
)
print(context_2)

test = context.accountant
test.output_measure = "MaxDivergence"

context_3 = dp.Context.compositor(
    **context_param,
    privacy_loss=dp.loss_of(epsilon=0.5),
    split_by_weights=[1],
    domain=
)

print(context_3)

Context(
    accountant = Measurement(
        input_domain   = FrameDomain(species: str, island: str, bill_length_mm: f64, bill_depth_mm: f64, flipper_length_mm: f64, body_mass_g: f64, sex: str; margins=[{}]),
        input_metric   = SymmetricDistance(),
        output_measure = ZeroConcentratedDivergence),
    d_in       = 1,
    d_mids     = [0.125, 0.375])
Context(
    accountant = Measurement(
        input_domain   = FrameDomain(species: str, island: str, bill_length_mm: f64, bill_depth_mm: f64, flipper_length_mm: f64, body_mass_g: f64, sex: str; margins=[{}]),
        input_metric   = SymmetricDistance(),
        output_measure = MaxDivergence),
    d_in       = 1,
    d_mids     = [0.5])


AttributeError: property 'output_measure' of 'Measurement' object has no setter

In [1]:
import polars as pl
import os 
import opendp.prelude as dp

dp.enable_features("contrib")

%pip install polars==1.12.0

storage_options = {
    "aws_access_key_id": os.environ["AWS_ACCESS_KEY_ID"],
    "aws_secret_access_key": os.environ["AWS_SECRET_ACCESS_KEY"],
    "aws_session_token": os.environ["AWS_SESSION_TOKEN"],
    "endpoint_url": "https://minio.lab.sspcloud.fr",
}

source = "s3://gferey/diffusion/synthetic-filo/METRO/population/population_METRO.parquet"


lf = pl.read_parquet(source, storage_options=storage_options).drop("geometry").lazy()

# Crée le premier context avec la mesure zCDP (rho)
context_1 = dp.Context.compositor(
    data = lf,
    privacy_unit = dp.unit_of(contributions=1),
    margins = [dp.polars.Margin(max_partition_length=1)],
    privacy_loss=dp.loss_of(rho=0.5),
    split_by_weights=[1, 3]
)



[notice] A new release of pip is available: 25.1.1 -> 25.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [3]:
lf_empty = pl.LazyFrame(schema=lf.collect_schema())

context_2_tmp = dp.Context.compositor(
    data=lf_empty,
    privacy_unit=dp.unit_of(contributions=1),
    margins=[dp.polars.Margin(max_partition_length=1)],
    privacy_loss=dp.loss_of(epsilon=0.5),
    split_by_weights=[1]
)

context_2 = dp.Context(
    accountant=context_2_tmp.accountant,
    queryable=context_1.queryable,
    d_in=context_2_tmp.d_in,
    d_mids=context_2_tmp.d_mids
)
print(context_2)

Context(
    accountant = Measurement(
        input_domain   = FrameDomain(ID: str, HOUSEHOLD_ID: str, HOUSEHOLD_SIZE: i64, GRD_MENAGE: bool, MONOPARENT: bool, NIVEAU_VIE: f64, TILE_ID: str, AGE_CAT: str, AGE: i64, ADULT: bool, STATUT: str; margins=[{}]),
        input_metric   = SymmetricDistance(),
        output_measure = MaxDivergence),
    d_in       = 1,
    d_mids     = [0.5])


In [11]:
print(context_1.queryable.value)

In [7]:
from src.request_class import Count, Sum, Mean, Ratio, Quantile

Quantile(variable = "NIVEAU_VIE", bounds=[0.0,300000.0],alpha=[0.5], nb_candidats="300").execute_dp(context_2, None)

OpenDPException: 
  MeasureMismatch("Intermediate measures don't match. See https://github.com/opendp/opendp/discussions/297
    output_measure: ZeroConcentratedDivergence
    input_measure:  MaxDivergence
")

In [ ]:
import polars as pl
import os 

storage_options = {
    "aws_access_key_id": os.environ["AWS_ACCESS_KEY_ID"],
    "aws_secret_access_key": os.environ["AWS_SECRET_ACCESS_KEY"],
    "aws_session_token": os.environ["AWS_SESSION_TOKEN"],
    "endpoint_url": "https://minio.lab.sspcloud.fr",
}

source = "s3://gferey/diffusion/synthetic-filo/METRO/population/population_METRO.parquet"


lf = pl.read_parquet(source, storage_options=storage_options)

# Crée le premier context avec la mesure zCDP (rho)
context_1 = dp.Context.compositor(
    data = lf,
    privacy_unit = dp.unit_of(contributions=1),
    margins = [dp.polars.Margin(max_partition_length=1)],
    privacy_loss=dp.loss_of(rho=0.5),
    split_by_weights=[1, 3],
    domain = context_1.accountant.input_domain
)

# On crée un LazyFrame vide avec le même schéma
lf_empty_same_schema = pl.LazyFrame(schema=lf.collect_schema())

# Réutilise le queryable (déjà construit) dans un second context avec une autre mesure
context_2 = dp.Context.compositor(
    data = lf_empty_same_schema,
    privacy_unit = dp.unit_of(contributions=1),
    margins = [dp.polars.Margin(max_partition_length=1)],
    privacy_loss=dp.loss_of(epsilon=0.5),
    split_by_weights=[1, 3]
)

context_3 = dp.Context(
    accountant=context_2.accountant,
    queryable=context_2.queryable,
    d_in=1,
    d_mids=[0.5]
)

print(context_1)
print(context_2)
print(context_3)


Context(
    accountant = Measurement(
        input_domain   = FrameDomain(species: str, island: str, bill_length_mm: f64, bill_depth_mm: f64, flipper_length_mm: f64, body_mass_g: f64, sex: str; margins=[{}]),
        input_metric   = SymmetricDistance(),
        output_measure = ZeroConcentratedDivergence),
    d_in       = 1,
    d_mids     = [0.125, 0.375])
Context(
    accountant = Measurement(
        input_domain   = FrameDomain(species: str, island: str, bill_length_mm: f64, bill_depth_mm: f64, flipper_length_mm: f64, body_mass_g: f64, sex: str; margins=[{}]),
        input_metric   = SymmetricDistance(),
        output_measure = MaxDivergence),
    d_in       = 1,
    d_mids     = [0.125, 0.375])
Context(
    accountant = Measurement(
        input_domain   = FrameDomain(species: str, island: str, bill_length_mm: f64, bill_depth_mm: f64, flipper_length_mm: f64, body_mass_g: f64, sex: str; margins=[{}]),
        input_metric   = SymmetricDistance(),
        output_measure = Max